<a href="https://colab.research.google.com/github/Naqeebullah11/BERT-masked-language-modeling/blob/main/BERT_Masked_Language_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT Masked Language Modeling
### Predicting Missing Words Using a Pretrained BERT Model

#Import Libraries

In [2]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForMaskedLM

# Load Tokenizer and Model

In [3]:
MODEL_NAME = "bert-base-uncased"

print("Loading model...")

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForMaskedLM.from_pretrained(MODEL_NAME)

model.eval()

print("Model loaded successfully!")


Loading model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


# Function to predict masked word


In [4]:
def predict_mask(sentence):

    # Check whether sentence contains [MASK]
    if tokenizer.mask_token not in sentence:
        raise ValueError(
            f"Sentence must contain {tokenizer.mask_token}"
        )

    # Tokenize
    inputs = tokenizer(sentence, return_tensors="pt")

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # Forward pass
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = outputs.logits

    # Find MASK position
    mask_index = torch.where(
        input_ids == tokenizer.mask_token_id
    )[1]

    # Predict token
    predicted_token_id = torch.argmax(
        logits[0, mask_index],
        dim=-1
    )

    predicted_word = tokenizer.convert_ids_to_tokens(
        predicted_token_id.item()
    )

    predicted_word = predicted_word.replace("##", "")

    # Replace MASK token
    input_ids[0, mask_index] = predicted_token_id

    completed_sentence = tokenizer.decode(
        input_ids[0],
        skip_special_tokens=True
    )

    return predicted_word, completed_sentence


In [5]:
examples = [
    "The movie was [MASK].",
    "Artificial Intelligence is [MASK].",
    "I love playing [MASK]."
]

print("\n======= EXAMPLES =======\n")

for sentence in examples:

    word, completed = predict_mask(sentence)

    print("Input Sentence :", sentence)
    print("Predicted Word :", word)
    print("Completed      :", completed)
    print("-"*60)



======= EXAMPLES =======

Input Sentence : The movie was [MASK].
Predicted Word : good
Completed      : the movie was good.
------------------------------------------------------------
Input Sentence : Artificial Intelligence is [MASK].
Predicted Word : possible
Completed      : artificial intelligence is possible.
------------------------------------------------------------
Input Sentence : I love playing [MASK].
Predicted Word : football
Completed      : i love playing football.
------------------------------------------------------------


# User Input

In [ ]:
print("\n======= USER INPUT =======\n")

while True:

    sentence = input("Enter a sentence containing [MASK] (or type 'exit'): ")

    if sentence.lower() == "exit":
        break

    if "[MASK]" not in sentence:
        print("\n Please include [MASK] in your sentence.")
        print("Example: The movie was [MASK].\n")
        continue

    try:

        word, completed = predict_mask(sentence)

        print("\nPredicted Word :", word)
        print("Completed      :", completed)
        print()

    except Exception as e:
        print("Error:", e)



======= USER INPUT =======

Enter a sentence containing [MASK] (or type 'exit'): The capital of France is [MASK].

Predicted Word : paris
Completed      : the capital of france is paris.

Enter a sentence containing [MASK] (or type 'exit'): The sun is [MASK].

Predicted Word : rising
Completed      : the sun is rising.

